# CLINICAL TRIALS RISK ANALYSIS
---
## 01 - Data Collection
This notebook performs the first step of the data pipeline: **retrieving raw clinical trial records using the modern ClinicalTrials.gov API (API v2), flattening each JSON study object into a row**, and saving the result as a **flattened raw dataset for downstream preprocessing**.

### Objective  
The objective of this step is to programmatically download the **first 100,000 clinical trial study records**, extract key fields from the hierarchical JSON structure, convert them into a uniform dataframe through a custom flattening function, and save them as **raw CSV files**.  
The flattened output is saved as a CSV file and will become the starting point for: data validation, exploratory data analysis (EDA), feature engineering, machine learning-based risk modeling.

### Why collect raw JSON?  
ClinicalTrials.gov API v2 provides a modernized schema where each study is represented as a deeply nested JSON object containing:

- **protocolSection** $\rightarrow$ primary study information  
  - **identificationModule** $\rightarrow$ NCT ID, titles, organization  
  - **statusModule** $\rightarrow$ recruitment status, key study dates  
  - **designModule** $\rightarrow$ study design, phase, allocation, model  
  - **conditionsModule** $\rightarrow$ conditions, keywords  
  - **armsInterventionsModule** $\rightarrow$ arms, interventions, drug/device info  
  - **outcomesModule** $\rightarrow$ primary/secondary outcomes  
  - **descriptionModule** $\rightarrow$ brief and detailed summaries  
  - **sponsorCollaboratorsModule** $\rightarrow$ lead sponsor, collaborators  
  - **eligibilityModule** $\rightarrow$ inclusion/exclusion criteria, sex, age  
  - **contactsLocationsModule** $\rightarrow$ study officials, locations  
  - **oversightModule** $\rightarrow$ oversight and data monitoring committee info  
  - **referencesModule** $\rightarrow$ PMIDs and literature references  
- **derivedSection** $\rightarrow$ standardized MeSH terms & curated metadata  
- **hasResults** $\rightarrow$ boolean flag indicating whether the study has posted results  

Working directly with the raw JSON ensures that no structural information is lost before flattening or feature extraction.

### Output  
Running this notebook will generate: `data/raw/clinical_trials_raw.csv` containing approximately **100,000 flattened study records**, which serves as the input for preprocessing, EDA, and risk-prediction modeling.

### Imports & Config

In [1]:
import sys
import os

# Path to project root (folder that contains "src/")
project_root = os.path.abspath("..")
sys.path.append(project_root)

masked = project_root.replace(os.path.expanduser("~"), "~")
print("Project root added:", masked)

Project root added: ~/Clinical-Trial-Failure-Prediction


In [2]:
import json
import requests
import pandas as pd

from src.data.data_loader import ClinicalTrialsAPI, flatten_study

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"
OUTPUT_PATH = "../data/raw/clinical_trials_raw.csv"

### Smoke Test: Fetch 1 Record

In [3]:
# Quick test request to check API availability & structure
response = requests.get(BASE_URL, params={"pageSize": 1})
response.raise_for_status()

sample = response.json()
sample.keys(), len(sample["studies"])

(dict_keys(['studies', 'nextPageToken']), 1)

### Inspect Structure of a Study (Schema Exploration)
ClinicalTrials.gov studies are highly heterogeneous. Many modules are optional, so downstream code must handle missing sections.

In [4]:
sample_study = sample["studies"][0]
print("Study keys:")
print(sample_study.keys())

protocol = sample_study.get("protocolSection", {})

print("protocolSection keys:")
print(list(protocol.keys()))

ident = protocol.get("identificationModule", {})
status = protocol.get("statusModule", {})
design = protocol.get("designModule", {})

print("\nIdentification Module:")
print(json.dumps(ident, indent=2))

print("\nStatus Module:")
print(json.dumps(status, indent=2))

print("\nDesign Module:")
print(json.dumps(design, indent=2))

Study keys:
dict_keys(['protocolSection', 'derivedSection', 'hasResults'])
protocolSection keys:
['identificationModule', 'statusModule', 'sponsorCollaboratorsModule', 'oversightModule', 'descriptionModule', 'conditionsModule', 'designModule', 'armsInterventionsModule', 'outcomesModule', 'eligibilityModule', 'contactsLocationsModule', 'ipdSharingStatementModule']

Identification Module:
{
  "nctId": "NCT05549635",
  "orgStudyIdInfo": {
    "id": "PFBIO-HP"
  },
  "organization": {
    "fullName": "University Hospital, Gentofte, Copenhagen",
    "class": "OTHER"
  },
  "briefTitle": "Database and Biobank of Patients With Hypersensitivity Pneumonitis",
  "officialTitle": "Pulmonary Fibrosis Biobank - Hypersensitivity Pneumonitis",
  "acronym": "PFBIO-HP"
}

Status Module:
{
  "statusVerifiedDate": "2023-12",
  "overallStatus": "RECRUITING",
  "expandedAccessInfo": {
    "hasExpandedAccess": false
  },
  "startDateStruct": {
    "date": "2022-09-18",
    "type": "ACTUAL"
  },
  "primaryCo

### Initialize API Loader (from data_loader.py)
A small sleep interval (0.3s) is used to avoid server-side throttling.

In [5]:
api = ClinicalTrialsAPI(page_size=100, sleep=0.3, max_retries=3)
api 

### Fetch Studies
Because the API returns data in pages of up to 1000 records per request, this operation will issue 100 API calls. A retry mechanism and short delay are used to ensure robust data collection.

In [6]:
studies = api.fetch_n_studies(n=100000)
len(studies)

[INFO] Fetching up to 100000 studies...
[INFO] Retrieved batch with 1000 studies (total: 1000)
[INFO] Retrieved batch with 1000 studies (total: 2000)
[INFO] Retrieved batch with 1000 studies (total: 3000)
[INFO] Retrieved batch with 1000 studies (total: 4000)
[INFO] Retrieved batch with 1000 studies (total: 5000)
[INFO] Retrieved batch with 1000 studies (total: 6000)
[INFO] Retrieved batch with 1000 studies (total: 7000)
[INFO] Retrieved batch with 1000 studies (total: 8000)
[INFO] Retrieved batch with 1000 studies (total: 9000)
[INFO] Retrieved batch with 1000 studies (total: 10000)
[INFO] Retrieved batch with 1000 studies (total: 11000)
[INFO] Retrieved batch with 1000 studies (total: 12000)
[INFO] Retrieved batch with 1000 studies (total: 13000)
[INFO] Retrieved batch with 1000 studies (total: 14000)
[INFO] Retrieved batch with 1000 studies (total: 15000)
[INFO] Retrieved batch with 1000 studies (total: 16000)
[INFO] Retrieved batch with 1000 studies (total: 17000)
[INFO] Retrieved 

100000

### Quick Sanity Check

In [7]:
# Show first 10 NCT IDs
ids = [
    s.get("protocolSection", {})
     .get("identificationModule", {})
     .get("nctId")
    for s in studies[:10]
]

ids

['NCT05549635',
 'NCT05414435',
 'NCT00516035',
 'NCT01143935',
 'NCT02728635',
 'NCT07131735',
 'NCT01338935',
 'NCT03480035',
 'NCT05353335',
 'NCT02557035']

### Check for Duplicate Studies

In [8]:
unique_ids = {
    s.get("protocolSection", {})
     .get("identificationModule", {})
     .get("nctId")
    for s in studies
}

print("Total:", len(studies))
print("Unique:", len(unique_ids))
print("Duplicates:", len(studies) - len(unique_ids))

Total: 100000
Unique: 100000
Duplicates: 0


### Save Raw Data

In [9]:
rows = [flatten_study(s) for s in studies]
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_PATH, index=False)

### Validate Output File

In [10]:
size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"Saved file size: {size_mb:.2f} MB")

Saved file size: 20.45 MB
